In [2]:
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.colors as mcolors

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_finelabels"

adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")

# FIX 1: drop the excluded (NaN) cells entirely before plotting, rather
# than letting scanpy plot them as a generic "NA" category
adata1_plot = adata1[adata1.obs["cell_type_fine"].notna()].copy()
adata1_plot.obs["cell_type_fine"] = adata1_plot.obs["cell_type_fine"].astype("category")
print(f"Plotting {adata1_plot.n_obs} cells ({adata1.n_obs - adata1_plot.n_obs} excluded, unlabeled)")

# FIX 2: build a colour map that groups related categories into
# visually distinct colour families (lymphoid=blues/purples,
# myeloid=greens/browns, etc.) rather than relying on the default
# palette, which produces too many near-identical shades at 24 categories
categories = sorted(adata1_plot.obs["cell_type_fine"].cat.categories.tolist())

color_map = {
    # CD4 states - blues
    "CD4 Naive/Resting T cells": "#08519c",
    "CD4 Activated T cells": "#3182bd",
    "Regulatory T cells (Tregs, CD4+)": "#6baed6",
    # CD8 states - purples
    "Naive/Memory CD8 T cells": "#54278f",
    "Activated CD8 T cells": "#756bb1",
    "Effector CD8 T cells": "#9e9ac8",
    "NK-like CD8 T cells": "#bcbddc",
    "Cycling CD8 T cells": "#dadaeb",
    # NK/NKT - pinks
    "True NK cells": "#c51b8a",
    "NKT cells": "#fa9fb5",
    "NK/Cytotoxic T cells": "#fde0dd",
    # Macrophages - greens
    "LAM-like macrophages": "#006d2c",
    "Lipid-laden/Foam-cell macrophages": "#31a354",
    "Antigen-presenting macrophages": "#74c476",
    "Complement-high macrophages": "#a1d99b",
    "Resting/Resident macrophages": "#c7e9c0",
    "Monocyte-like macrophages": "#238b45",
    "Non-classical monocytes (CD16+)": "#41ae76",
    # Other immune - oranges/browns
    "B cells": "#d94801",
    "Monocytes/DC": "#fd8d3c",
    "Mast cells": "#fdae6b",
    "pDC": "#e6550d",
    # Contamination/artefacts - greys
    "Mixed/stromal-contaminated (CD8+fibroblast signal)": "#969696",
    "Unassigned (n=91, stromal/RBC contamination artefact)": "#bdbdbd",
    "Non-T-cell contamination (from T cells parent cluster)": "#d9d9d9",
}
# Fallback for any category not explicitly mapped
palette = [color_map.get(cat, "#000000") for cat in categories]

fig, ax = plt.subplots(figsize=(13, 10))
sc.pl.umap(adata1_plot, color="cell_type_fine",
           title="GSE114725 — cell_type_fine (fine-resolution annotation)",
           palette=palette,
           legend_loc="right margin", legend_fontsize=7,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_celltype_fine_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: GSE114725_celltype_fine_umap.png (corrected)")

Plotting 43516 cells (1146 excluded, unlabeled)
Saved: GSE114725_celltype_fine_umap.png (corrected)


In [5]:
import os
FIGURE_DIR_CHECK = PROJECT_DIR / "figures"
for root, dirs, files in os.walk(FIGURE_DIR_CHECK):
    for f in files:
        if "cd8" in f.lower():
            print(os.path.join(root, f))

C:\Users\annam\Dissertation 2026\figures\phase3_pathway_enrichment\GSE176078_DE_CD8_T_cells_TNBC_vs_ER+_pathway_dotplot.png
C:\Users\annam\Dissertation 2026\figures\phase3_pathway_enrichment\GSE176078_DE_CD8_T_cells_TNBC_vs_HER2+_pathway_dotplot.png
C:\Users\annam\Dissertation 2026\figures\phase3_pseudobulk_de\GSE114725_volcano_CD8_Effector_T_cells_tumor_vs_normal.png
C:\Users\annam\Dissertation 2026\figures\phase3_pseudobulk_de\GSE176078_volcano_CD8_T_cells_HER2+_vs_ER+.png
C:\Users\annam\Dissertation 2026\figures\phase3_pseudobulk_de\GSE176078_volcano_CD8_T_cells_TNBC_vs_ER+.png
C:\Users\annam\Dissertation 2026\figures\phase3_pseudobulk_de\GSE176078_volcano_CD8_T_cells_TNBC_vs_HER2+.png


In [7]:
# ----------------------------
# Corrected palette — distinct hues for a standalone CD8 figure,
# rather than the monochromatic purple family used in the full
# cell_type_fine UMAP
# ----------------------------
cd8_colors_distinct = {
    "Naive/Memory CD8 T cells": "#1f77b4",   # blue
    "Activated CD8 T cells": "#d62728",       # red
    "Effector CD8 T cells": "#2ca02c",        # green
    "NK-like CD8 T cells": "#ff7f0e",         # orange
    "Cycling CD8 T cells": "#9467bd",         # purple
}
palette_cd8_distinct = [cd8_colors_distinct[c] for c in sorted(adata1_cd8.obs["cd8_subtype_v2"].unique())]

fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.umap(adata1_cd8, color="cd8_subtype_v2",
           title="GSE114725 — CD8 T cell sub-clusters",
           palette=palette_cd8_distinct,
           legend_loc="right margin", legend_fontsize=9,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_CD8_subclusters_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: GSE114725_CD8_subclusters_umap.png (corrected palette)")

Saved: GSE114725_CD8_subclusters_umap.png (corrected palette)


In [9]:
# ----------------------------
# Filter zero-variance genes before HVG/scaling — same fix as the
# LIANA investigation. This CD8-only subset apparently has some genes
# with no variance across all 10,916 cells.
# ----------------------------
X_check = adata2_cd8.X
if hasattr(X_check, "toarray"):
    X_check = X_check.toarray()
variances = X_check.var(axis=0)
n_zero_var = (variances == 0).sum()
print(f"Zero-variance genes in this subset: {n_zero_var}")

if n_zero_var > 0:
    keep_genes = adata2_cd8.var_names[variances != 0]
    adata2_cd8 = adata2_cd8[:, keep_genes].copy()
    print(f"Filtered to {adata2_cd8.n_vars} genes")

# Now proceed with HVG/scaling/PCA as before
sc.pp.highly_variable_genes(adata2_cd8, n_top_genes=2000, flavor="seurat")
adata2_cd8_hvg = adata2_cd8[:, adata2_cd8.var.highly_variable].copy()
sc.pp.scale(adata2_cd8_hvg, max_value=10)
sc.tl.pca(adata2_cd8_hvg, svd_solver="arpack", random_state=0)

import scanpy.external as sce
sce.pp.harmony_integrate(adata2_cd8_hvg, key="orig.ident", basis="X_pca", random_state=0)
sc.pp.neighbors(adata2_cd8_hvg, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.umap(adata2_cd8_hvg, random_state=42)

adata2_cd8_hvg.obs["cell_type_fine"] = adata2_cd8.obs["cell_type_fine"].values
print("Ready to plot")

Zero-variance genes in this subset: 0


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_highly_variable_genes.py:336: RuntimeWarning: invalid value encountered in log
  dispersion = np.log(dispersion)
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: `n_top_genes` > number of normalized dispersions, returning all genes with normalized dispersions.
  return fn(*args_all, **kw)
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_scale.py:199: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)


ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [11]:
cd8_colors_176078 = {
    "Resting/Memory-like CD8 T cells": "#1f77b4",
    "Stress-Response/Activated CD8 T cells": "#d62728",
    "Effector CD8 T cells": "#2ca02c",
    "GZMK+ CD8 T cells": "#ff7f0e",
    "Exhausted CD8 T cells": "#9467bd",
    "Interferon-Response CD8 T cells": "#17becf",
}
palette_176078 = [cd8_colors_176078[c] for c in sorted(adata2_cd8_simple.obs["cell_type_fine"].unique())]

fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.umap(adata2_cd8_simple, color="cell_type_fine",
           title="GSE176078 — CD8 T cell sub-clusters",
           palette=palette_176078,
           legend_loc="right margin", legend_fontsize=8,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE176078_CD8_subclusters_umap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: GSE176078_CD8_subclusters_umap.png")

Saved: GSE176078_CD8_subclusters_umap.png


In [12]:
adata1_tcells = sc.read_h5ad(PROCESSED_DIR / "GSE114725_tcells_subclustered_v2.h5ad")
print(f"Loaded: {adata1_tcells.n_obs} cells")
print(adata1_tcells.obs["tcell_subtype_v2"].value_counts())
print("X_umap already present:", "X_umap" in adata1_tcells.obsm)
print("X_pca_harmony already present:", "X_pca_harmony" in adata1_tcells.obsm)

Loaded: 20213 cells
tcell_subtype_v2
CD4 Activated T cells                                     8532
CD4 Naive/Resting T cells                                 7555
Regulatory T cells (Tregs, CD4+)                          2992
CD8 T cells (reclassified from T cells parent cluster)    1090
Non-T-cell contamination (from T cells parent cluster)      44
Name: count, dtype: int64
X_umap already present: True
X_pca_harmony already present: True


In [13]:
tcell_colors = {
    "CD4 Naive/Resting T cells": "#1f77b4",
    "CD4 Activated T cells": "#d62728",
    "Regulatory T cells (Tregs, CD4+)": "#2ca02c",
    "CD8 T cells (reclassified from T cells parent cluster)": "#ff7f0e",
    "Non-T-cell contamination (from T cells parent cluster)": "#7f7f7f",
}
palette_tcell = [tcell_colors[c] for c in sorted(adata1_tcells.obs["tcell_subtype_v2"].unique())]

fig, ax = plt.subplots(figsize=(9, 8))
sc.pl.umap(adata1_tcells, color="tcell_subtype_v2",
           title="GSE114725 — T cell sub-clusters (corrected)",
           palette=palette_tcell,
           legend_loc="right margin", legend_fontsize=8,
           frameon=True, ax=ax, show=False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_tcell_subclusters_umap_v2.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: GSE114725_tcell_subclusters_umap_v2.png")

Saved: GSE114725_tcell_subclusters_umap_v2.png
